# 10c: GO + Disease Enrichment for Differential Accessibility

Runs after 10b_differential_accessibility.ipynb and 10b_enterocytes.ipynb.
Loads saved ultra-robust peak sets and runs: GO (BP), KEGG, Disease Ontology, DisGeNET, Cancer Genes.

In [ ]:
# Cell 1: Packages & utilities
suppressPackageStartupMessages({
  library(DESeq2)
  library(clusterProfiler)
  library(DOSE)
  library(org.Hs.eg.db)
  library(enrichplot)
  library(ggplot2)
  library(dplyr)
  library(readr)
  library(tibble)
})
source("../src/deseq2_utils.R")
message("Packages loaded.")

In [ ]:
# Cell 2: Configuration
BASE    <- "/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
OUT_DIR <- file.path(BASE, "cross_species_consensus_v3/13_deseq2_R_pseudobulk")
SPECIES <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")

save_dir <- file.path(OUT_DIR, "pseudobulk")
summary_dir <- file.path(OUT_DIR, "_summary")

# Load master annotation (for gene lookup)
anno_file   <- file.path(BASE, "cross_species_consensus_v3/07_master_annotation/master_annotation_final.tsv")
master_anno <- read_tsv(anno_file, show_col_types = FALSE)
options(scipen = 999)

message("Config OK. Annotation: ", nrow(master_anno), " peaks")

In [ ]:
# Cell 3: Load ultra-robust evolutionary peaks + shared-peak DESeq2 results
robust_peaks  <- readRDS(file.path(summary_dir, "ultra_robust_peaks_list.rds"))
res_shared    <- readRDS(file.path(summary_dir, "DESeq2_res_list_shared.rds"))

message("Ultra-robust peak sets:")
for (ct in names(robust_peaks)) {
  n <- sum(sapply(robust_peaks[[ct]], length))
  message("  ", ct, ": ", n, " peaks across ", length(robust_peaks[[ct]]), " contrasts")
}

# Also load branch_res_list for universe extraction in evolutionary enrichment
branch_res_path <- file.path(summary_dir, "branch_res_list.rds")
if (file.exists(branch_res_path)) {
  branch_res <- readRDS(branch_res_path)
  message("branch_res loaded: ", length(branch_res), " cell types")
} else {
  branch_res <- NULL
  warning("branch_res_list.rds not found — enrichment will use default universe")
}


## Evolutionary contrast enrichment (all cell types)

Runs GO + KEGG + Disease Ontology + DisGeNET + Cancer Genes for each cell type × key evolutionary contrast.

In [ ]:
# Cell 4: Full enrichment — evolutionary contrasts, all cell types
# Universe = all peaks tested per cell type × contrast (from branch_res).
key_contrasts <- c(
  "Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys",
  "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates",
  "ILS_HumanGorilla_vs_Pan", "ILS_HumanChimp_vs_GorillaBonobo",
  "ILS_HumanBonobo_vs_ChimpGorilla"
)

for (ct in names(robust_peaks)) {
  message("\n========== ", ct, " ==========")
  ct_contrasts <- intersect(key_contrasts, names(robust_peaks[[ct]]))
  if (length(ct_contrasts) == 0) {
    message("  No matching contrasts, skipping.")
    next
  }
  for (cn in ct_contrasts) {
    peaks <- robust_peaks[[ct]][[cn]]
    # Universe: all peaks tested for this cell type × contrast
    uni_peaks <- if (!is.null(branch_res) && !is.null(branch_res[[ct]][[cn]]))
      rownames(as.data.frame(branch_res[[ct]][[cn]])) else NULL
    run_full_enrichment(
      peaks          = peaks,
      species        = "Human",
      label          = paste0(ct, "_", cn),
      out_dir        = OUT_DIR,
      anno_df        = master_anno,
      ct             = ct,
      universe_peaks = uni_peaks
    )
  }
}
message("\n=== Evolutionary contrast enrichment complete ===")


## Species-specific enrichment (Human + Marmoset focus)

Uses the shared-peak all-vs-one results to extract species-specific UP peaks and run disease enrichment.

In [ ]:
# Cell 5: Full enrichment — species-specific UP peaks from shared-peak DESeq2
# Universe = all peaks tested for that cell type × species (rownames of full DESeq2 result).
species_focus <- c("Human", "Marmoset", "Macaque", "Gorilla")

for (ct in names(res_shared)) {
  message("\n--- ", ct, " ---")
  for (sp in intersect(species_focus, names(res_shared[[ct]]))) {
    df     <- as.data.frame(res_shared[[ct]][[sp]])
    up_pk  <- rownames(df)[!is.na(df$padj) & df$padj < 0.05 & df$log2FoldChange > 1]
    run_full_enrichment(
      peaks          = up_pk,
      species        = "Human",
      label          = paste0(ct, "_SpecificUP_", sp),
      out_dir        = OUT_DIR,
      anno_df        = master_anno,
      ct             = ct,
      universe_peaks = rownames(df)
    )
  }
}
message("\n=== Species-specific enrichment complete ===")


## Disease summary across all contrasts

Collect top DisGeNET and Disease Ontology terms across all contrasts into a single summary table — useful for identifying which intestinal diseases are most enriched.

In [ ]:
# Cell 6: Aggregate disease enrichment summary across all cell types
# Files are now at: {OUT_DIR}/{CellType}/enrichment/{label}_top_disease_terms.csv
library(dplyr)
summary_dir <- file.path(OUT_DIR, "_summary")
dir.create(summary_dir, showWarnings = FALSE, recursive = TRUE)

# Search recursively for all _top_disease_terms.csv files
top_files <- list.files(OUT_DIR, pattern = "_top_disease_terms\\.csv$",
                        recursive = TRUE, full.names = TRUE)
message("Found ", length(top_files), " disease term summary files")

if (length(top_files) > 0) {
  all_terms <- lapply(top_files, read.csv) |> bind_rows()

  # Focus on DisGeNET + Disease Ontology
  disease_summary <- all_terms |>
    filter(Database %in% c("DisGeNET", "DiseaseOntology")) |>
    group_by(Database, Description) |>
    summarise(
      n_contrasts    = n(),
      min_padj       = min(p.adjust, na.rm = TRUE),
      contrasts      = paste(Label, collapse = "; "),
      .groups = "drop"
    ) |>
    arrange(Database, desc(n_contrasts), min_padj)

  write.csv(disease_summary,
            file.path(summary_dir, "SUMMARY_disease_terms_across_contrasts.csv"),
            row.names = FALSE)

  message("Top recurrent disease associations (DisGeNET + DO):")
  print(head(disease_summary, 30))

  # KEGG summary
  kegg_summary <- all_terms |>
    filter(Database == "KEGG") |>
    group_by(Description) |>
    summarise(n_contrasts = n(), min_padj = min(p.adjust, na.rm = TRUE),
              contrasts = paste(Label, collapse = "; "), .groups = "drop") |>
    arrange(desc(n_contrasts), min_padj)

  write.csv(kegg_summary,
            file.path(summary_dir, "SUMMARY_KEGG_pathways_across_contrasts.csv"),
            row.names = FALSE)
  message("\nTop recurrent KEGG pathways:")
  print(head(kegg_summary, 20))
} else {
  message("No enrichment summary files found yet. Run enrichment cells first.")
}


In [ ]:
# Cell 7: Final summary
message("\n=== Enrichment Notebook Complete ===")
message("Results saved to:")
message("  Enrichment: ", OUT_DIR, "/{CellType}/enrichment/")
message("  Summary:    ", file.path(OUT_DIR, "_summary/"))